# Analysis for the unconditional generation

In [ ]:
from matplotlib import pyplot as plt
from plotly import express as px
from tqdm import tqdm
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
from tools import (
    compute_uniqueness,
    compute_novelty,
    compute_unique_novelty,
)

## Load data

In [ ]:
df_eqgat = pd.read_csv("predictions/unconditional/eqgat/eqgat_100000_predictions.csv")
df_eqgat["method"] = "EQGAT"
# df_gcdm = pd.read_csv("predictions/unconditional/gcdm/gcdm_100000_predictions.csv")
# df_gcdm["method"] = "GCDM-SBDD"
# df_geoldm = pd.read_csv("predictions/unconditional/geoldm/geoldm_100000_predictions.csv")
# df_geoldm["method"] = "GeoLDM"
df_semla = pd.read_csv(
    "predictions/unconditional/semlaflow/semlaflow_100000_predictions.csv"
)
df_semla["method"] = "SemlaFlow"
df_flowmol = pd.read_csv(
    "predictions/unconditional/molflow/molflow_100000_predictions.csv"
)[:100000]
df_flowmol["method"] = "MolFlow"

df_train = pd.read_csv("data/unconditional/geom-drugs/train.csv")
df_train["method"] = "GEOM Drugs Training"
# df_val = pd.read_csv("predictions_data/val.csv")
# df_val["method"] = "GEOM Drugs Validation"
# df_test = pd.read_csv("predictions_data/test.csv")
# df_test["method"] = "GEOM Drugs Testing"

df = pd.concat([df_flowmol, df_semla, df_eqgat, df_train])

# Evaluation script does not break molecules correctly, introducting some rows
df = df[~df.fail.fillna(0).astype(bool)]


In [ ]:
# define order of methods
order = [
    "GEOM Drugs Training",
    # "GEOM Drugs Validation",
    # "GEOM Drugs Testing",
    "MolFlow",
    "SemlaFlow",
    "EQGAT",
    "GCDM-SBDD",
    "GeoLDM",
]
df["method"] = pd.Categorical(df["method"], categories=order, ordered=True)

In [ ]:
# enrichment
reference_smiles = set(df_train["smiles"].values)
df["total_number"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]
df["novel"] = df["smiles"].map(lambda x: x not in reference_smiles)
df["valid_novel"] = df["valid"] & df["novel"]
df["valid_smiles"] = df["valid"].astype(bool) * df["smiles"]

## Tables

### Validity

In [ ]:
def mean(x):
    n = 100000
    if len(x) == 0:
        return float("nan")
    if len(x) > n:
        return np.mean(x)
    return np.sum(x) / n


def std(x):
    n = 100000
    if len(x) == 0:
        return float("nan")
    if len(x) > n:
        return np.std(x)
    m = np.sum(x) / n
    return np.sqrt(np.sum((x - m) ** 2) / n)


aggs = {
    "total_number": ("total_number", "sum"),
    "generated": ("total_number", mean),
    "connected": ("connected", mean),
    "chemical": ("chemical", mean),
    "physical": ("physical", mean),
    "valid": ("valid", mean),
}
df_agg = df.groupby("method", observed=False).agg(**aggs)
cols = df_agg.columns
df_agg.style.format("{:.0f}", subset=cols[:1]).format("{:.1%}", subset=cols[1:])

### Properties of valid molecules

In [ ]:
df_filter = df[df.valid]
aggs = {
    "qed": ["mean", "std"],
    "sa": ["mean", "std"],
    "energy_ratio": ["mean", "std"],
    "weight": ["mean", "std"],
    "num_heavy": ["mean", "std"],
    "num_rings": ["mean", "std"],
    "lipinski": ["mean", "std"],
    "logp": ["mean", "std"],
    "spacial": ["mean", "std"],
}
df_agg = df_filter.groupby("method", observed=False).agg(aggs)
cols = df_agg.columns
df_agg.style.format("{:.2f}", subset=cols)

### Energy ratio

In [ ]:
# df_filter = df[df.valid]
df_filter = df
aggs = {
    "ensemble_avg_energy": ["mean", "std"],
    "mol_pred_energy": ["mean", "std"],
    "energy_ratio": ["mean", "std"],
}
df_agg = df_filter.groupby("method", observed=False).agg(aggs)
cols = df_agg.columns
df_agg.style.format("{:.2f}", subset=cols)

### Uniqueness and novelty

In [ ]:
n = 100000

df_filter = df
aggs = {
    "Valid": ("valid", lambda x: sum(x) / n if len(x) <= n else sum(x) / 240988),
    "Valid & Unique": (
        "valid_smiles",
        lambda x: compute_uniqueness(x, total=1) / n
        if len(x) <= n
        else compute_uniqueness(x, total=1) / 240988,
    ),
    "Valid & Novel": (
        "valid_smiles",
        lambda x: compute_novelty(x, reference_smiles, total=1) / n,
    ),
    "Valid & Unique & Novel": (
        "valid_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=1) / n,
    ),
}
df_agg = df_filter.groupby("method", observed=False).agg(**aggs)
cols = df_agg.columns
df_agg.style.format("{:.2%}", subset=cols)

## Plots

In [ ]:
metrics = {
    "ensemble_avg_energy": "Ensemble Average Energy",
    "mol_pred_energy": "Molecular Prediction Energy",
    "energy_ratio": "Energy Ratio",
    "sa": "Synthetic Accessability Score",
    "sa_normalized": "Synthetic Accessability Score (normalized)",
    "spacial": "Spacial Score",
    "qed": "Quantitative Estimation of Drug-likeness",
    "logp": "LogP",
    "lipinski": "Lipinski Rule of 5",
    "num_heavy": "Number of Heavy Atoms",
    "weight": "Molecular Weight",
    "num_rings": "Number of Rings",
}

### Synthetically accessibility

In [ ]:
metric = "sa"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 10)
plt.savefig(f"plots/unconditional_{metric}.png")

### Spacial score

In [ ]:
metric = "spacial"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 120)

### QED

In [ ]:
metric = "qed"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 1)
plt.savefig(f"plots/unconditional_{metric}.png")

### Energy ratio

In [ ]:
metric = "energy_ratio"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    bins=100,
    hue="method",
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    log_scale=True if metric == "energy_ratio" else False,
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0.5, 200)
plt.savefig(f"plots/unconditional_{metric}.png")

### Number of heavy atoms

In [ ]:
metric = "num_heavy"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    bins=np.array(range(int(df[df.valid]["num_heavy"].max()) + 1)),
    hue="method",
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    fill=True,
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 60)
plt.savefig(f"plots/unconditional_{metric}.png")

# Appendix

In [ ]:
metric = "logp"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    bins=100,
    # bins=np.array(range(int(df[df.valid]["num_heavy"].max()) + 1)),
    hue="method",
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    fill=True,
    # legend=True, palette="tab10", linewidth=1.5
)
# plt.title(name)
plt.xlabel(name)
# plt.xlim(0, 60)
plt.savefig(f"plots/unconditional_{metric}.png")

In [ ]:
# plot all metrics and save a plot
for metric, name in tqdm(metrics.items()):
    sns.histplot(
        df[df.valid][["method", metric]].reset_index(drop=True),
        x=metric,
        hue="method",
        cumulative=False,
        common_norm=False,
        stat="density",
        element="step",
        # legend=True, palette="tab10", linewidth=1.5
    )
    plt.title(name)
    plt.xlabel(name)
    plt.savefig(f"plots/unconditonal/plot_{metric}.png")
    plt.close()

### Old tables

In [ ]:
df_filter = df[df.valid]
aggs = {
    "Novel": ("smiles", lambda x: compute_novelty(x, reference_smiles)),
    "Unique": ("smiles", compute_uniqueness),
    "Unique novel": (
        "smiles",
        lambda x: compute_unique_novelty(x, reference_smiles),
    ),
}
df_agg = df_filter.groupby("method", observed=False).agg(**aggs)
cols = df_agg.columns
df_agg.style.format("{:.2%}", subset=cols)


In [ ]:
df_filter = df
aggs = {
    "Valid": ("valid_smiles", len),
    "(%)": ("valid_smiles", lambda x: 1.0),
    "Unique & Valid": (
        "valid_smiles",
        lambda x: compute_uniqueness(x, total=1),
    ),
    "(%) ": (
        "valid_smiles",
        lambda x: compute_uniqueness(x, total=len(x)),
    ),
    "Novel and Valid": (
        "valid_smiles",
        lambda x: compute_novelty(x, reference_smiles, total=1),
    ),
    "(%)  ": (
        "valid_smiles",
        lambda x: compute_novelty(x, reference_smiles, total=len(x)),
    ),
    "Unique & Novel & Valid)": (
        "valid_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=1),
    ),
    "(%)   ": (
        "valid_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=len(x)),
    ),
}
df_agg = df_filter.groupby("method", observed=False).agg(**aggs)
cols = df_agg.columns
df_agg.style.format("{:.0f}", subset=cols[0]).format(
    "{:.2%}", subset=[cols[1], cols[3], cols[5], cols[5], cols[7]]
).format("{:.0f}", subset=[cols[2], cols[4], cols[6]])